### Step 1: Prepare Documents

In [ ]:
import json

with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)

In [ ]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict
        documents.append(doc)
        
documents[1]

### Step 2: Create Embeddings using Pretrained Models

In [ ]:
# pip install sentence_transformers==2.7.0
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-mpnet-base-v2')

In [ ]:
len(model.encode('This is a simple sentence'))

In [ ]:
#  created the dense vector using the pre-trained model
operations = []
for doc in documents:
    #operations.append({"index": {"_index": "doc_ix"}})
    # Transforming the title into an embedding using the model
    doc['text_vector'] = model.encode(doc['text']).tolist()
    operations.append(doc)

In [ ]:
operations[1]

### Step 3: Setup ElasticSearch connection

In [ ]:
from elasticsearch import Elasticsearch
es_client = Elasticsearch('http://localhost:9200')

In [ ]:
es_client.info()

### Step 4: Create Mappings and Index
    - Mapping is the process of defining how a document, and the fields it contains, are stored and indexed.
    - Each document is a collecton of fields, which each have their own data type.
    - We can compare mapping to a database schema in how it describes the fields and properties that documents hold, the datatype of each field (eg., string, integer, or date), and how those fields should be indexed and stored.

In [ ]:
index_settings = {
    "settings": {
        "number_of_shards":1,
        "number_of_replicas":0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"},
            "text_vector": {"type": "dense_vector", "dims":768, "index":True, "similarity": "cosine"
                            },
        }
    }
}

In [ ]:
index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailability=True)
es_client.indices.create(index=index_name, body=index_settings)

### Step 5: Add documents into index

In [ ]:
for doc in operations:
    try:
        es_client.index(index=index_name, document=doc)
    except Exception as e:
        print(e)

### Step 6: Create end user query

In [ ]:
search_term = "windows or mac?"
vector_search_term = model.encode(search_term)

In [ ]:
query = {
    "field": "text_vector",
    "query_vector" : vector_search_term,
    "k": 5,
    "num_candidates": 10000,
}

In [ ]:
res = es_client.search(index=index_name, knn=query,source=["text","section","question","course"])
res["hits"]["hits"]

### Step 7: Perform Semantic Search & Advanced Search

In [ ]:
# response = es_client.search(
#     index=index_name,
#     query={
#         "bool": {
#             "must": {
#              "multi_match": 
#                         {"query": "windows or python?", 
#                          "fields": ["text", "question","course","title"],
#                          "type": "best_fields"
#                         }
#                     },
#             "filter": {
#                 "term": {
#                         "course": "data-engineering-zoomcamp"
#             }
#         }
#         }
#     }
# )

In [ ]:
# response['hits']['hits']

In [ ]:
knn_query = {
    "field": "text_vector",
    "query_vector": vector_search_term,
    "k": 5,
    "num_candidates": 10000
}

response = es_client.search(
    index=index_name,
    query={
        'match': {
            'course':'data-engineering-zoomcamp'
        },
    },
    
    knn=knn_query,
    size=5,
    explain=True # tells you how the scores are calculated
)